In [1]:
"""
Примеры кода "Философия LangChain и основы LCEL"

Этот модуль демонстрирует:
1. Базовые Runnable-компоненты
2. Pipe-оператор и RunnableSequence
3. RunnableParallel для ветвления
4. RunnablePassthrough и RunnableLambda
5. Визуализация и отладка цепочек
"""

'\nПримеры кода "Философия LangChain и основы LCEL"\n\nЭтот модуль демонстрирует:\n1. Базовые Runnable-компоненты\n2. Pipe-оператор и RunnableSequence\n3. RunnableParallel для ветвления\n4. RunnablePassthrough и RunnableLambda\n5. Визуализация и отладка цепочек\n'

In [ ]:
import os
from typing import Any
from dotenv import load_dotenv

In [3]:
# Загрузка переменных окружения
load_dotenv()

True

# ============================================================================
# ЧАСТЬ 1: БАЗОВЫЕ RUNNABLE-КОМПОНЕНТЫ
# ============================================================================

In [7]:
from langchain_core.runnables import Runnable, RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
"""
Демонстрация базового интерфейса Runnable.

Каждый компонент LangChain реализует интерфейс Runnable,
который определяет методы invoke, batch, stream.
"""
print("=" * 60)
print("ПРИМЕР 1: Базовый интерфейс Runnable")
print("=" * 60)

# Создаём простейшую модель
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="google/gemini-3-flash-preview",
    temperature=0,
)

# --- Метод invoke - базовый синхронный вызов
print("\n1.1. Метод invoke() - единичный вызов:")
result = model.invoke("Привет! Скажи одно слово.")
print(f"Тип результата: {type(result).__name__}")
print(f"Ответ: {result.content}")

# --- Метод batch - обработка нескольких входов
print("\n1.2. Метод batch() - пакетная обработка:")
inputs = ["Назови цвет неба одним словом", "Назови цвет травы одним словом", "Назови цвет солнца одним словом",]
results = model.batch(inputs)
for inp, res in zip(inputs, results):
    print(f"  '{inp[:30]}...' -> '{res.content}'")

# --- Метод stream - потоковая генерация
print("\n1.3. Метод stream() - потоковый вывод:")
print("  Ответ: ", end="")
for chunk in model.stream("Напиши короткое хайку о Python"):
    print(chunk.content, end="", flush=True)
print()

ПРИМЕР 1: Базовый интерфейс Runnable

1.1. Метод invoke() - единичный вызов:
Тип результата: AIMessage
Ответ: Привет!

1.2. Метод batch() - пакетная обработка:
  'Назови цвет неба одним словом...' -> 'Голубой'
  'Назови цвет травы одним словом...' -> 'Зелёный'
  'Назови цвет солнца одним слово...' -> 'Белый.'

1.3. Метод stream() - потоковый вывод:
  Ответ: Змея в коде спит,
Чистый синтаксис манит,
Мир в одной строке.


In [ ]:
"""
Языковая модель как Runnable.

ChatOpenAI принимает:
- Строку (автоматически оборачивается в HumanMessage)
- Список сообщений
- Объект сообщения
"""
print("\n" + "=" * 60)
print("ПРИМЕР 2: Модель как Runnable")
print("=" * 60)

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="google/gemini-3-flash-preview",
    temperature=0,
)

# --- Способ 1: Простая строка
print("\n2.1. Вызов со строкой:")
result1 = model.invoke("Что такое Python?")
print(f"  {result1.content[:100]}...")

# --- Способ 2: Список сообщений
print("\n2.2. Вызов со списком сообщений:")
messages = [
    SystemMessage(content="Ты — эксперт по программированию. Отвечай кратко."),
    HumanMessage(content="Что такое Python?"),
]
result2 = model.invoke(messages)
print(f"  {result2.content[:100]}...")

# --- Способ 3: Сообщение напрямую
print("\n2.3. Вызов с одним сообщением:")
result3 = model.invoke([HumanMessage(content="Что такое Python?")])
# result3 = model.invoke("Что такое Python?")
print(f"  {result3.content[:100]}...")


ПРИМЕР 2: Модель как Runnable

2.1. Вызов со строкой:
  **Python** (читается как «пайтон» или «питон») — это высокоуровневый язык программирования общего на...

2.2. Вызов со списком сообщений:
  **Python** — это высокоуровневый язык программирования общего назначения, ориентированный на читаемо...

2.3. Вызов с одним сообщением:
  **Python** (читается как «пайтон» или «питон») — это высокоуровневый язык программирования общего на...


In [19]:
"""
Шаблон промпта как Runnable.

ChatPromptTemplate принимает словарь с переменными
и возвращает форматированные сообщения.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 3: Промпт как Runnable")
print("=" * 60)

# Создаём шаблон
prompt = ChatPromptTemplate.from_messages(
    [("system", "Ты — эксперт в области {domain}. Отвечай кратко и по делу."), ("human", "{question}")]
)

# Инспекция схемы входа
print("\n3.1. Схема входных данных:")
print(f"  {prompt.input_schema.model_json_schema()}")

# Вызов промпта
print("\n3.2. Форматирование промпта:")
formatted = prompt.invoke({"domain": "физики", "question": "Что такое энтропия?"})
print(f"  Тип результата: {type(formatted).__name__}")
print(f"  Количество сообщений: {len(formatted.messages)}")
for msg in formatted.messages:
    print(f"  [{msg.__class__.__name__}]: {msg.content[:50]}...")


ПРИМЕР 3: Промпт как Runnable

3.1. Схема входных данных:
  {'properties': {'domain': {'title': 'Domain', 'type': 'string'}, 'question': {'title': 'Question', 'type': 'string'}}, 'required': ['domain', 'question'], 'title': 'PromptInput', 'type': 'object'}

3.2. Форматирование промпта:
  Тип результата: ChatPromptValue
  Количество сообщений: 2
  [SystemMessage]: Ты — эксперт в области физики. Отвечай кратко и по...
  [HumanMessage]: Что такое энтропия?...


# ============================================================================
# ЧАСТЬ 2: PIPE-ОПЕРАТОР И RUNNABLESEQUENCE
# ============================================================================

In [20]:
"""
Демонстрация pipe-оператора (|) для соединения компонентов.

Оператор | создаёт RunnableSequence, где выход предыдущего
компонента становится входом следующего.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 4: Pipe-оператор")
print("=" * 60)

# Компоненты
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Ты — лаконичный ассистент. Отвечай одним предложением."),
        ("human", "Объясни, что такое {concept}"),
    ]
)
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-5-mini",
    temperature=0,
)
parser = StrOutputParser()

# Создание цепочки через pipe
chain = prompt | model | parser

print("\n4.1. Тип созданной цепочки:")
print(f"  {type(chain).__name__}")

print("\n4.2. Вызов цепочки:")
result = chain.invoke({"concept": "рекурсия"})
print(f"  Результат: {result}")

# Пошаговая демонстрация
print("\n4.3. Пошаговое выполнение:")
step1 = prompt.invoke({"concept": "рекурсия"})
print(f"  Шаг 1 (prompt): {step1.messages[1].content}")
step2 = model.invoke(step1)
print(f"  Шаг 2 (model): {step2.content[:80]}...")
step3 = parser.invoke(step2)
print(f"  Шаг 3 (parser): {step3[:80]}...")


ПРИМЕР 4: Pipe-оператор

4.1. Тип созданной цепочки:
  RunnableSequence

4.2. Вызов цепочки:
  Результат: Рекурсия — метод в математике и программировании, при котором функция или определение решает задачу, вызывая само себя на меньших (или упрощённых) данных и имеет базовый случай для прекращения вызовов.

4.3. Пошаговое выполнение:
  Шаг 1 (prompt): Объясни, что такое рекурсия
  Шаг 2 (model): Рекурсия — это приём, когда задача или функция решается/определяется через более...
  Шаг 3 (parser): Рекурсия — это приём, когда задача или функция решается/определяется через более...


In [21]:
"""
RunnableSequence — класс, создаваемый оператором pipe.

Можно создать его явно или через оператор |.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 5: RunnableSequence")
print("=" * 60)


from langchain_core.runnables import RunnableSequence


prompt = ChatPromptTemplate.from_template("Переведи на английский: {text}")
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)
parser = StrOutputParser()

# Способ 1: Через pipe
chain1 = prompt | model | parser

# Способ 2: Явное создание
chain2 = RunnableSequence(first=prompt, middle=[model], last=parser)

print("\n5.1. Сравнение способов создания:")
print(f"  chain1 type: {type(chain1).__name__}")
print(f"  chain2 type: {type(chain2).__name__}")

# Оба дают одинаковый результат
result1 = chain1.invoke({"text": "Привет, мир!"})
result2 = chain2.invoke({"text": "Привет, мир!"})
print(f"\n5.2. Результаты идентичны: {result1 == result2}")
print(f"  Результат: {result1}")


ПРИМЕР 5: RunnableSequence

5.1. Сравнение способов создания:
  chain1 type: RunnableSequence
  chain2 type: RunnableSequence

5.2. Результаты идентичны: True
  Результат: Hello, world!


# ============================================================================
# ЧАСТЬ 3: RUNNABLEPARALLEL — ПАРАЛЛЕЛЬНОЕ ВЕТВЛЕНИЕ
# ============================================================================

In [23]:
"""
RunnableParallel выполняет несколько компонентов параллельно
с одним и тем же входом.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 6: RunnableParallel")
print("=" * 60)


from langchain_core.runnables import RunnableParallel


model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)

# Создаём параллельные ветки
parallel = RunnableParallel(
    {
        "summary": ChatPromptTemplate.from_template("Кратко опиши в одном предложении: {text}")
        | model
        | StrOutputParser(),
        "keywords": ChatPromptTemplate.from_template("Выдели 3 ключевых слова из текста (через запятую): {text}")
        | model
        | StrOutputParser(),
        "sentiment": ChatPromptTemplate.from_template(
            "Определи тональность текста одним словом (позитивная/негативная/нейтральная): {text}"
        )
        | model
        | StrOutputParser(),
    }
)

text = """Python — замечательный язык программирования!
Он простой для изучения, но при этом очень мощный.
Многие разработчики любят его за читаемый синтаксис."""

print("\n6.1. Параллельная обработка текста:")
result = parallel.invoke({"text": text})

print(f"  Summary: {result['summary']}")
print(f"  Keywords: {result['keywords']}")
print(f"  Sentiment: {result['sentiment']}")



ПРИМЕР 6: RunnableParallel

6.1. Параллельная обработка текста:
  Summary: Python — это мощный и простой в изучении язык программирования с читаемым синтаксисом, который пользуется популярностью среди разработчиков.
  Keywords: Python, язык программирования, синтаксис
  Sentiment: Позитивная.


In [24]:
"""
Короткий синтаксис для RunnableParallel — словарь в цепочке.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 7: Короткий синтаксис RunnableParallel")
print("=" * 60)

from langchain_core.runnables import RunnablePassthrough

model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)

# Словарь в цепочке автоматически становится RunnableParallel
chain = {
    "original": RunnablePassthrough(),  # Пропускаем вход как есть
    "upper": RunnableLambda(lambda x: x.upper()),
    "length": RunnableLambda(lambda x: len(x)),
} | RunnableLambda(lambda d: f"Original: {d['original']}, Upper: {d['upper']}, Length: {d['length']}")

print("\n7.1. Цепочка с параллельной обработкой:")
result = chain.invoke("hello world")
print(f"  Результат: {result}")


ПРИМЕР 7: Короткий синтаксис RunnableParallel

7.1. Цепочка с параллельной обработкой:
  Результат: Original: hello world, Upper: HELLO WORLD, Length: 11


# ============================================================================
# ЧАСТЬ 4: RUNNABLEPASSTHROUGH И RUNNABLELAMBDA
# ============================================================================

In [27]:
"""
RunnablePassthrough пропускает вход без изменений.

Особенно полезен в RunnableParallel, когда нужно
сохранить исходные данные наряду с обработанными.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 8: RunnablePassthrough")
print("=" * 60)


from langchain_core.runnables import RunnablePassthrough, RunnableParallel


model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)

# Паттерн RAG: сохраняем вопрос + добавляем контекст
def fake_retriever(question: str) -> str:
    """Имитация retriever'а."""
    return "Python создан Гвидо ван Россумом в 1991 году."

prompt = ChatPromptTemplate.from_messages(
    [("system", "Отвечай на основе контекста: {context}"), ("human", "{question}")]
)

chain = (
    {"context": RunnableLambda(fake_retriever), "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

print("\n8.1. RAG-паттерн с RunnablePassthrough:")
result = chain.invoke("Кто создал Python?")
print(f"  Ответ: {result}")


ПРИМЕР 8: RunnablePassthrough

8.1. RAG-паттерн с RunnablePassthrough:
  Ответ: Python был создан Гвидо ван Россумом в 1991 году.


In [ ]:
"""
RunnablePassthrough.assign() добавляет поля к входному словарю.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 9: RunnablePassthrough.assign()")
print("=" * 60)

from langchain_core.runnables import RunnablePassthrough
from datetime import datetime

# Добавляем поля к входным данным
enricher = RunnablePassthrough.assign(
    timestamp=lambda _: datetime.now().isoformat(), 
    word_count=lambda x: len(x.get("text", "").split()),
)

print("\n9.1. Обогащение данных:")
input_data = {"text": "Hello world from Python", "author": "User"}
enriched = enricher.invoke(input_data)
print(f"  Исходные данные: {input_data}")
print(f"  Обогащённые данные: {enriched}")


ПРИМЕР 9: RunnablePassthrough.assign()

9.1. Обогащение данных:
  Исходные данные: {'text': 'Hello world from Python', 'author': 'User'}
  Обогащённые данные: {'text': 'Hello world from Python', 'author': 'User', 'timestamp': '2026-02-09T04:22:12.941858', 'word_count': 4}


In [29]:
"""
RunnableLambda оборачивает произвольную Python-функцию в Runnable.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 10: RunnableLambda")
print("=" * 60)

# Способ 1: Оборачивание функции
def extract_first_word(text: str) -> str:
    return text.split()[0] if text else ""

first_word = RunnableLambda(extract_first_word)

print("\n10.1. RunnableLambda из функции:")
result = first_word.invoke("Hello World Python")
print(f"  Результат: {result}")

# Способ 2: Лямбда-функция напрямую
word_count = RunnableLambda(lambda x: len(x.split()))
print(f"\n10.2. RunnableLambda из lambda:")
result = word_count.invoke("Hello World Python")
print(f"  Количество слов: {result}")

# Способ 3: Декоратор @chain
from langchain_core.runnables import chain

@chain
def process_text(text: str) -> dict:
    """Обработка текста."""
    words = text.split()
    return {"original": text, "word_count": len(words), "first_word": words[0] if words else None}

print("\n10.3. RunnableLambda через декоратор @chain:")
result = process_text.invoke("Python is awesome")
print(f"  Результат: {result}")


ПРИМЕР 10: RunnableLambda

10.1. RunnableLambda из функции:
  Результат: Hello

10.2. RunnableLambda из lambda:
  Количество слов: 3

10.3. RunnableLambda через декоратор @chain:
  Результат: {'original': 'Python is awesome', 'word_count': 3, 'first_word': 'Python'}


# ============================================================================
# ЧАСТЬ 5: ВИЗУАЛИЗАЦИЯ И ОТЛАДКА
# ============================================================================

In [32]:
"""
Инспекция схем входа и выхода компонентов.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 11: Инспекция схем")
print("=" * 60)

prompt = ChatPromptTemplate.from_messages([("system", "Ты — эксперт в {domain}"), ("human", "{question}")])
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
)
parser = StrOutputParser()

chain = prompt | model | parser

print("\n11.1. Схема входа цепочки:")
print(f"  {chain.input_schema.schema()}")

print("\n11.2. Схема выхода цепочки:")
print(f"  {chain.output_schema.schema()}")

print("\n11.3. Структура графа выполнения:")
try:
    graph = chain.get_graph()
    print(f"  Узлы: {[node.id for node in graph.nodes.values()]}")
    print(f"  Рёбра: {[(e.source, e.target) for e in graph.edges]}")
except Exception as e:
    print(f"  (Упрощённый вывод: {e})")


ПРИМЕР 11: Инспекция схем

11.1. Схема входа цепочки:
  {'properties': {'domain': {'title': 'Domain', 'type': 'string'}, 'question': {'title': 'Question', 'type': 'string'}}, 'required': ['domain', 'question'], 'title': 'PromptInput', 'type': 'object'}

11.2. Схема выхода цепочки:
  {'title': 'StrOutputParserOutput', 'type': 'string'}

11.3. Структура графа выполнения:
  Узлы: ['a907c0e648284349b48c7a3244ebe897', '7b1c6abe533a4d19b42cfffda40ea50d', '3b290d28a64a4ff79e2baf1a9c27f2c8', '0436d79a02384b8aaa9cad7742d0a76c', '5fb7adfc2c074813a513314a74c5c527']
  Рёбра: [('a907c0e648284349b48c7a3244ebe897', '7b1c6abe533a4d19b42cfffda40ea50d'), ('7b1c6abe533a4d19b42cfffda40ea50d', '3b290d28a64a4ff79e2baf1a9c27f2c8'), ('0436d79a02384b8aaa9cad7742d0a76c', '5fb7adfc2c074813a513314a74c5c527'), ('3b290d28a64a4ff79e2baf1a9c27f2c8', '0436d79a02384b8aaa9cad7742d0a76c')]


/var/folders/d_/k331795x38q58drfdtws24lh0000gn/T/ipykernel_90619/2171368468.py:19: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(f"  {chain.input_schema.schema()}")


In [ ]:
"""
Трейсинг выполнения цепочки через callbacks.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 12: Трейсинг выполнения")
print("=" * 60)

from langchain_core.callbacks import BaseCallbackHandler
from datetime import datetime

class SimpleTracer(BaseCallbackHandler):
    """Простой трейсер для отладки."""

    def on_chain_start(self, serialized, inputs, **kwargs):
        print(f"  [CHAIN START] inputs={inputs}")

    def on_chain_end(self, outputs, **kwargs):
        print(f"  [CHAIN END] outputs={str(outputs)[:80]}...")

    def on_llm_start(self, serialized, prompts, **kwargs):
        print(f"  [LLM START] {len(prompts)} prompt(s)")

    def on_llm_end(self, response, **kwargs):
        print(f"  [LLM END] generated response")


prompt = ChatPromptTemplate.from_template("Скажи '{word}' по-английски")
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)
chain = prompt | model | StrOutputParser()

print("\n12.1. Выполнение с трейсингом:")
tracer = SimpleTracer()
result = chain.invoke({"word": "привет"}, config={"callbacks": [tracer]})
print(f"\n  Финальный результат: {result}")


ПРИМЕР 12: Трейсинг выполнения

12.1. Выполнение с трейсингом:
  [CHAIN START] inputs={'word': 'привет'}
  [CHAIN START] inputs={'word': 'привет'}
  [CHAIN END] outputs=messages=[HumanMessage(content="Скажи 'привет' по-английски", additional_kwargs=...
  [LLM START] 1 prompt(s)
  [LLM END] generated response
  [CHAIN START] inputs=content='"Привет" по-английски будет "hello".' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 19, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 1.125e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.125e-05, 'upstream_inference_prompt_cost': 2.85e-06, 'upstream_inference_completions_cost': 8.4e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint':

In [36]:
"""
Использование встроенного ConsoleCallbackHandler.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 13: ConsoleCallbackHandler")
print("=" * 60)


from langchain_core.tracers import ConsoleCallbackHandler


prompt = ChatPromptTemplate.from_template("Что такое {topic}? Ответь кратко.")
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)
chain = prompt | model | StrOutputParser()

print("\n13.1. Выполнение с ConsoleCallbackHandler:")
print("-" * 40)
result = chain.invoke(
    input={"topic": "LCEL"},
    config={"callbacks": [ConsoleCallbackHandler()]},
)
print("-" * 40)
print(f"Результат: {result[:100]}...")


ПРИМЕР 13: ConsoleCallbackHandler

13.1. Выполнение с ConsoleCallbackHandler:
----------------------------------------
[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "topic": "LCEL"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "topic": "LCEL"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: Что такое LCEL? Ответь кратко."
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatOpenAI] [1.63s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "LCEL (Low-Cost Energy Lab) — это лаборатория или инициатива, направленная на разработку и внедрение технологий для получения и использования энергии с низкими затратами. Основная цель LCEL — сделать энергетику более доступной и устойчивой, используя инновационные по

# ============================================================================
# ЧАСТЬ 6: КОМПЛЕКСНЫЙ ПРИМЕР
# ============================================================================

In [37]:
"""
Комплексный пример: цепочка для анализа и улучшения текста.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 14: Комплексная цепочка")
print("=" * 60)


from langchain_core.runnables import RunnablePassthrough, RunnableParallel


model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="gpt-4o-mini",
    temperature=0,
)

# Шаг 1: Параллельный анализ
analysis = RunnableParallel(
    {
        "text": RunnablePassthrough(),
        "grammar_issues": (
            ChatPromptTemplate.from_template(
                "Найди грамматические ошибки в тексте (если нет, напиши 'нет ошибок'): {text}"
            )
            | model
            | StrOutputParser()
        ),
        "style_issues": (
            ChatPromptTemplate.from_template(
                "Найди стилистические проблемы (если нет, напиши 'нет проблем'): {text}"
            )
            | model
            | StrOutputParser()
        ),
    }
)

# Шаг 2: Улучшение на основе анализа
improvement_prompt = ChatPromptTemplate.from_template(
    """Исходный текст: {text}

Грамматические замечания: {grammar_issues}
Стилистические замечания: {style_issues}

Перепиши текст, исправив найденные проблемы. Если проблем нет, верни исходный текст."""
)

# Полная цепочка
chain = analysis | improvement_prompt | model | StrOutputParser()

test_text = "Програмисты пишет код на языке питон очень медлено."

print("\n14.1. Исходный текст:")
print(f"  '{test_text}'")

print("\n14.2. Результат обработки:")
result = chain.invoke({"text": test_text})
print(f"  '{result}'")



ПРИМЕР 14: Комплексная цепочка

14.1. Исходный текст:
  'Програмисты пишет код на языке питон очень медлено.'

14.2. Результат обработки:
  '"Программисты пишут код на языке Python очень медленно."'
